# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedwaqasahmad/FlyRank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule: a page is worth reviewing if it's stale (days_since_last_update >= 180) AND still visible (impressions_90d >= 500) — the "stale_visible_page" flag confirmed in ML-06 to have a 94.1% churn rate, far above the 54.2% base rate. I'll also add two supporting reason codes for broader coverage since the strict combo only catches 17 pages: "declining_with_demand" (already churned, still getting traffic) and "page_one_decay_risk" (ranks well but is aging).

In [5]:
import os, subprocess
REPO_URL = "https://github.com/syedwaqasahmad/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

import pandas as pd, numpy as np, os as _os
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_churned"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Reason codes:")
print("- stale_visible_page: days_since_last_update>=180 AND impressions_90d>=500")
print("- declining_with_demand: trend_direction=='down' AND impressions_90d>=100")
print("- page_one_decay_risk: avg_position<=10 AND content_age_days>=180")

Reason codes:
- stale_visible_page: days_since_last_update>=180 AND impressions_90d>=500
- declining_with_demand: trend_direction=='down' AND impressions_90d>=100
- page_one_decay_risk: avg_position<=10 AND content_age_days>=180


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Scoring every page using a weighted combination of the three signals, ranking all pages, and writing the result to work/outputs/baseline_action_score.csv.

In [6]:
df["reason_stale_visible"] = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).astype(int)
df["reason_declining_demand"] = ((df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 100)).astype(int)
df["reason_position_decay"] = ((df["avg_position"] <= 10) & (df["content_age_days"] >= 180)).astype(int)

df["baseline_action_score"] = (
    0.5 * df["reason_stale_visible"] +
    0.3 * df["reason_declining_demand"] +
    0.2 * df["reason_position_decay"]
)

def reason_code(row):
    codes = []
    if row["reason_stale_visible"]: codes.append("stale_visible_page")
    if row["reason_declining_demand"]: codes.append("declining_with_demand")
    if row["reason_position_decay"]: codes.append("page_one_decay_risk")
    return ",".join(codes) if codes else "none"

df["reason_code"] = df.apply(reason_code, axis=1)

ranked = df.sort_values("baseline_action_score", ascending=False)
_os.makedirs("work/outputs", exist_ok=True)
ranked[["content_id","baseline_action_score","reason_code","is_churned","impressions_90d",
        "days_since_last_update","avg_position","trend_direction"]].to_csv(
    "work/outputs/baseline_action_score.csv", index=False)

print("Saved", len(ranked), "ranked rows to work/outputs/baseline_action_score.csv")
print(ranked[["content_id","baseline_action_score","reason_code","is_churned"]].head(10))

Saved 30000 ranked rows to work/outputs/baseline_action_score.csv
                 content_id  baseline_action_score  \
22872  content_e3ff1b093148                    1.0   
26840  content_7f116ae1f6f5                    1.0   
7452   content_72496874f806                    1.0   
12045  content_c2d929d83eaa                    0.8   
20837  content_928af3e22c80                    0.8   
5327   content_fe16a55cd13d                    0.8   
26810  content_ecb6215e79fd                    0.8   
3507   content_074ba6ead17b                    0.8   
21268  content_0a91db491d14                    0.8   
11489  content_5feee3994adb                    0.8   

                                             reason_code  is_churned  
22872  stale_visible_page,declining_with_demand,page_...           1  
26840  stale_visible_page,declining_with_demand,page_...           1  
7452   stale_visible_page,declining_with_demand,page_...           1  
12045           stale_visible_page,declining_with_deman

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Reviewing the top 20 ranked pages: what action a reviewer should take, why (reason code), how confident I am, and what would make this pick wrong.

In [7]:
top20 = ranked.head(20)[["content_id","baseline_action_score","reason_code","is_churned","impressions_90d"]]
top20["action"] = "Review for refresh"
top20["confidence_note"] = top20["reason_code"].apply(
    lambda r: "High — multiple reasons agree" if "," in r else "Medium — single signal"
)
top20["what_would_make_wrong"] = "If the page's decline is seasonal/temporary rather than structural, or if impressions_90d is inflated by a one-off spike."
top20

,content_id,baseline_action_score,reason_code,is_churned,impressions_90d,action,confidence_note,what_would_make_wrong
22872,content_e3ff1b093148,1.0,"stale_visible_page,declining_with_demand,page_...",1,1408,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
26840,content_7f116ae1f6f5,1.0,"stale_visible_page,declining_with_demand,page_...",1,954,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
7452,content_72496874f806,1.0,"stale_visible_page,declining_with_demand,page_...",1,821,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
12045,content_c2d929d83eaa,0.8,"stale_visible_page,declining_with_demand",1,7558,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
20837,content_928af3e22c80,0.8,"stale_visible_page,declining_with_demand",1,1697,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
5327,content_fe16a55cd13d,0.8,"stale_visible_page,declining_with_demand",1,4556,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
26810,content_ecb6215e79fd,0.8,"stale_visible_page,declining_with_demand",1,4429,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
3507,content_074ba6ead17b,0.8,"stale_visible_page,declining_with_demand",1,533,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
21268,content_0a91db491d14,0.8,"stale_visible_page,declining_with_demand",1,13299,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...
11489,content_5feee3994adb,0.8,"stale_visible_page,declining_with_demand",1,7812,Review for refresh,High — multiple reasons agree,If the page's decline is seasonal/temporary ra...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Checking whether any top-20 picks look wrong, and confirming no leakage — trend_pct and any product decision flags are absent from the scoring formula entirely.

In [8]:
weak_picks = top20[top20["is_churned"] == 0]
print(f"Top-20 picks that did NOT actually churn: {len(weak_picks)}")
print(weak_picks[["content_id","baseline_action_score","reason_code"]])

score_inputs = ["reason_stale_visible","reason_declining_demand","reason_position_decay"]
leaked = [c for c in score_inputs if "trend_pct" in c or "flag" in c.lower()]
print("\nLeakage check — any label-derived or product-flag inputs in the score formula:", leaked if leaked else "None found")

Top-20 picks that did NOT actually churn: 0
Empty DataFrame
Columns: [content_id, baseline_action_score, reason_code]
Index: []

Leakage check — any label-derived or product-flag inputs in the score formula: None found


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.